# Week 12: Applied Machine Learning Capstone

                **Focus:** Bring the course together with clean end-to-end regression and classification mini-projects.

                **Learning objectives**
                - Build a portfolio-ready workflow with preprocessing and evaluation
- Compare at least two models on the same split
- Write short conclusions about results and next steps

                **Source materials:** Sample_Data_Analysis.pdf, Data Analysis.pptx, CT4103 - Data Handling(1).pdf, Data Handling Session new.pptx, Intro to ML.pptx, housing.csv, framingham.csv

## Key Highlights

- Weeks 8-12 are organised by inferred topic progression from the supplied later materials
- End-to-end housing regression workflow
- End-to-end Framingham classification workflow

## Project setup

In [1]:
from pathlib import Path
import sys

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "requirements.txt").exists() and (candidate / "weeks").exists():
            return candidate
    return current

ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

ROOT


WindowsPath('F:/Recovery Asus TFug/Master Degree/1º semester/Python Notebooks and Scripting/Python Notebook/ct7201-guide')

## Housing regression mini-project

In [2]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

housing = pd.read_csv(ROOT / "data" / "housing.csv")
X_h = housing.drop(columns=["median_house_value"])
y_h = housing["median_house_value"]

num_cols = X_h.drop(columns=["ocean_proximity"]).columns.tolist()
cat_cols = ["ocean_proximity"]

preprocessor = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])

X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(X_h, y_h, test_size=0.2, random_state=42)
linear = Pipeline([("preprocessor", preprocessor), ("model", LinearRegression())]).fit(X_train_h, y_train_h)
forest = Pipeline([("preprocessor", preprocessor), ("model", RandomForestRegressor(n_estimators=80, random_state=42))]).fit(X_train_h, y_train_h)

linear_pred = linear.predict(X_test_h)
forest_pred = forest.predict(X_test_h)

display(pd.DataFrame({
    "model": ["Linear Regression", "Random Forest"],
    "mae": [mean_absolute_error(y_test_h, linear_pred), mean_absolute_error(y_test_h, forest_pred)],
    "r2": [r2_score(y_test_h, linear_pred), r2_score(y_test_h, forest_pred)],
}))


,model,mae,r2
0,Linear Regression,50670.489236,0.625438
1,Random Forest,31687.088975,0.817114


## Framingham classification mini-project

In [3]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

framingham = pd.read_csv(ROOT / "data" / "framingham.csv")
X_f = framingham.drop(columns=["TenYearCHD"])
y_f = framingham["TenYearCHD"]

preprocessor = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(X_f, y_f, test_size=0.2, random_state=42, stratify=y_f)

logistic = Pipeline([("preprocessor", preprocessor), ("model", LogisticRegression(max_iter=1000))]).fit(X_train_f, y_train_f)
forest = Pipeline([("preprocessor", preprocessor), ("model", RandomForestClassifier(n_estimators=120, random_state=42))]).fit(X_train_f, y_train_f)

log_pred = logistic.predict(X_test_f)
forest_pred = forest.predict(X_test_f)

display(pd.DataFrame({
    "model": ["Logistic Regression", "Random Forest"],
    "accuracy": [accuracy_score(y_test_f, log_pred), accuracy_score(y_test_f, forest_pred)],
    "precision": [precision_score(y_test_f, log_pred, zero_division=0), precision_score(y_test_f, forest_pred, zero_division=0)],
    "recall": [recall_score(y_test_f, log_pred, zero_division=0), recall_score(y_test_f, forest_pred, zero_division=0)],
}))


,model,accuracy,precision,recall
0,Logistic Regression,0.844340,0.411765,0.054264
1,Random Forest,0.847877,0.500000,0.038760


## Practice Tasks

- Compare baselines: Train at least two models and compare metrics fairly.
- Write conclusions: Summarise preprocessing choices, results, and recommended next steps.